# CS680: Assignment 1

## Exercise 1.1

Run your perceptron algorithm on the spambase dataset and plot the number of mistakes (y-axis) wrt the number of passes (x-axis).

---

Let's start by loading the dataset spambase_y.csv that contains the labels for the spambase dataset.

In [ ]:
import os
from CS680.assignments.a1.utils.ReadData import ReadY
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

y = ReadY("./a1/datasets/spambase_y.csv")


Now, let's manipulate the spambase_X.csv, so we can have feature vectors for each training example.

In [ ]:
spambase_x_df = pd.read_csv("./a1/datasets/spambase_X.csv", header=None)

spambase_x_df = spambase_x_df.T
X = spambase_x_df.to_numpy()

Function to train the perceptron that will also compute the number of mistakes for each pass.

In [ ]:
def train_perceptron(X, y, epochs=500):
    mistakes_vector = list()

    d = X.shape[1]

    w = np.zeros(d)
    b = 0.0

    for epoch in range(epochs):
        mistakes = 0

        for x_i, y_i in zip(X, y):

            score = np.dot(x_i, w) + b

            if y_i * score <= 0:
                w += y_i * x_i

                b += y_i

                mistakes += 1

        if mistakes == 0:
            break

        mistakes_vector.append(mistakes)

    return w, b, mistakes_vector

Let's train the perceptron and graph the number of mistakes for each pass.

In [ ]:
w, b, mistakes = train_perceptron(X, y)

plt.plot(range(1, len(mistakes) + 1), mistakes)

print(f"Weights: {w}")
print(f"Bias: {b:.2f}")
plt.xlabel("Number of passes")
plt.ylabel("Number of mistakes")
plt.title("Perceptron on Spambase")

plt.show()

---

## Exercise 1.2

Prove that in each iteration, the (binary) perceptron algorithm essentially picks a term from the above summation, computes the corresponding derivative (say g), and performs a gradient update.

Perceptron loss function for one training example

$$
\ell_i(w) = \max(0, -y_i x_i^T w)
$$

The idea of this interpretation is that we're using the gradient to update the weights. This way we'll be minimizing the weights. The algorithm behavior essentially does not change with the gradient update. This is just the proof of the perceptron algorithm.

There are two cases.

### Case 1: Correctly classified

If

$$
y_i x_i^T w > 0
$$

then

$$
-y_i x_i^T w < 0
$$

so

$$
\ell_i(w) = 0
$$

Therefore, the gradient is

$$
g = \nabla \ell_i(w) = 0
$$

and the update is

$$
w \leftarrow w - g = w
$$

so the weights do not change.

### Case 2: Misclassified

If

$$
y_i x_i^T w < 0
$$

then

$$
\ell_i(w) = -y_i x_i^T w
$$

The gradient with respect to $w$ is

$$
g = \nabla \ell_i(w) = -y_i x_i
$$

The gradient update is

$$
w \leftarrow w - g
$$

Substituting the gradient,

$$
w \leftarrow w - (-y_i x_i)
$$

therefore,

$$
\boxed{w \leftarrow w + y_i x_i}
$$

This is exactly the standard perceptron update rule.

Therefore, in each iteration, the perceptron selects one term from

$$
\sum_{i=1}^{n} \max(0, -y_i x_i^T w)
$$

computes its gradient, and performs a gradient update.

---

## Exercise 1.3

The multiclass perceptron objective is

$$
\min_{w_1,\ldots,w_c}
\sum_{i=1}^{n}
\max_{k=1,\ldots,c}
\left[
\langle x_i,w_k\rangle
-
\langle x_i,w_{y_i}\rangle
\right]
$$

When $c=2$, there are only two weight vectors $w_1$ and $w_2$

Define a single weight vector

$$
w = w_1-w_2.
$$

We consider the two possible classes.

### Case 1: $y_i=1$

The loss for example $i$ is

$$
\max
\left\{
\langle x_i,w_1\rangle-\langle x_i,w_1\rangle,
\;
\langle x_i,w_2\rangle-\langle x_i,w_1\rangle
\right\}.
$$

Therefore,

$$
=
\max
\left\{
0,
\langle x_i,w_2-w_1\rangle
\right\}.
$$

Since $w=w_1-w_2$,

$$
=
\max\{0,-\langle x_i,w\rangle\}.
$$

If we represent class 1 by the binary label $y_i=+1$, this becomes

$$
\max\{0,-y_i\langle x_i,w\rangle\}.
$$

### Case 2: $y_i=2$

The loss is

$$
\max
\left\{
\langle x_i,w_1\rangle-\langle x_i,w_2\rangle,
\;
\langle x_i,w_2\rangle-\langle x_i,w_2\rangle
\right\}.
$$

Therefore,

$$
=
\max\{0,\langle x_i,w_1-w_2\rangle\}.
$$

Using $w=w_1-w_2$,

$$
=
\max\{0,\langle x_i,w\rangle\}.
$$

If we represent class 2 by the binary label $y_i=-1$, then

$$
\max\{0,\langle x_i,w\rangle\}
=
\max\{0,-y_i\langle x_i,w\rangle\}.
$$

Therefore, for both classes, the objective becomes

$$
\boxed{
\min_w
\sum_{i=1}^{n}
\max\{0,-y_i\langle x_i,w\rangle\}
}
$$

where $y_i\in\{+1,-1\}$.

This is exactly the binary perceptron objective. Therefore, when $c=2$,
the multiclass perceptron reduces to the binary perceptron.

---

## Exercise 1.4

Develop and implement a multiclass perceptron algorithm. Run your implementation on the activity dataset and report the final errors on the training and test sets.

In [ ]:
X_train = np.loadtxt("./a1/datasets/activity_X_train.txt")
y_train = np.loadtxt("./a1/datasets/activity_y_train.txt", dtype=int)

X_test = np.loadtxt("./a1/datasets/activity_X_test.txt")
y_test = np.loadtxt("./a1/datasets/activity_y_test.txt", dtype=int)

In [ ]:
def train_multiclass_perceptron(X, y, num_passes=20):
    n_samples, n_features = X.shape

    classes = np.unique(y)
    n_classes = len(classes)

    class_to_index = {
        label: index
        for index, label in enumerate(classes)
    }

    W = np.zeros((n_classes, n_features))

    mistakes_per_pass = []

    for _ in range(num_passes):
        mistakes = 0

        for x_i, y_i in zip(X, y):

            scores = W @ x_i

            predicted_index = np.argmax(scores)

            true_index = class_to_index[y_i]

            if predicted_index != true_index:
                W[true_index] += x_i

                W[predicted_index] -= x_i

                mistakes += 1

        mistakes_per_pass.append(mistakes)

    return W, classes, mistakes_per_pass

In [ ]:
def predict_multiclass(X, W, classes):
    predictions = []

    for x_i in X:
        scores = W @ x_i
        predicted_index = np.argmax(scores)
        predictions.append(classes[predicted_index])

    return np.array(predictions)

In [ ]:
def classification_error(y_true, y_pred):
    return np.mean(y_true != y_pred)

In [ ]:
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

print(f"Classes: {np.unique(y_train)}")

In [ ]:
def train_multiclass_perceptron(X, y, num_passes=20):
    n_samples, n_features = X.shape

    classes = np.unique(y)
    n_classes = len(classes)

    W = np.zeros((n_classes, n_features))

    mistakes_per_pass = []

    for epoch in range(num_passes):
        mistakes = 0

        for x_i, y_i in zip(X, y):

            # Compute score for each class
            scores = W @ x_i

            # argmax gives an index: 0, 1, ..., 5
            predicted_index = np.argmax(scores)

            # Labels are 1, 2, ..., 6
            true_index = y_i - 1

            if predicted_index != true_index:
                # Increase score of correct class
                W[true_index] += x_i

                # Decrease score of incorrectly predicted class
                W[predicted_index] -= x_i

                mistakes += 1

        mistakes_per_pass.append(mistakes)

        print(
            f"Pass {epoch + 1}: "
            f"{mistakes} mistakes"
        )

    return W, mistakes_per_pass

In [ ]:
def predict_multiclass(X, W):
    predictions = []

    for x_i in X:
        scores = W @ x_i

        predicted_index = np.argmax(scores)

        # Convert index 0..5 back into label 1..6
        predicted_label = predicted_index + 1

        predictions.append(predicted_label)

    return np.array(predictions)

In [ ]:
def classification_error(y_true, y_pred) -> float:
    return float(np.mean(y_true != y_pred))

In [ ]:
W, mistakes = train_multiclass_perceptron(
    X_train,
    y_train,
    num_passes=20
)

In [ ]:
y_train_pred = predict_multiclass(X_train, W)
y_test_pred = predict_multiclass(X_test, W)

train_error = classification_error(
    y_train,
    y_train_pred
)

test_error = classification_error(
    y_test,
    y_test_pred
)

print(f"Training error: {train_error:.4f}")
print(f"Test error: {test_error:.4f}")

print(f"Training error: {train_error * 100:.2f}%")
print(f"Test error: {test_error * 100:.2f}%")

---

## Exercise 2.1

Show the (binary) square loss is proper.

We want to prove that the expected square loss is minimized when our predicted probability $p$ is equal to the true probability $q$.

We are given

$$
q = \Pr(Y=1).
$$

Since $Y$ is binary,

$$
\Pr(Y=0)=1-q.
$$

Therefore, $Y$ can only take the values

$$
Y=1
$$

or

$$
Y=0.
$$

We want to solve

$$
\min_p \mathbb E(Y-p)^2.
$$

First, we write the expected loss by considering both possible outcomes of $Y$:

$$
L(p)
=
\mathbb E(Y-p)^2
=
\Pr(Y=1)(1-p)^2
+
\Pr(Y=0)(0-p)^2.
$$

Substituting the probabilities,

$$
L(p)
=
q(1-p)^2
+
(1-q)(0-p)^2.
$$

Since $(0-p)^2=p^2$,

$$
L(p)
=
q(1-p)^2+(1-q)p^2.
$$

Expanding,

$$
L(p)
=
q(1-2p+p^2)+(1-q)p^2,
$$

$$
L(p)
=
q-2qp+qp^2+p^2-qp^2.
$$

Therefore,

$$
L(p)=q-2qp+p^2.
$$

To find the value of $p$ that minimizes the expected loss, we differentiate with respect to $p$:

$$
\frac{dL}{dp}=2p-2q.
$$

Setting the derivative equal to zero,

$$
2p-2q=0,
$$

which gives

$$
p=q.
$$

Finally,

$$
\frac{d^2L}{dp^2}=2>0,
$$

so $L(p)$ is strictly convex and $p=q$ is the unique minimum.

Therefore, the expected square loss is uniquely minimized when the predicted probability $p$ equals the true probability $q$.

Hence, the binary square loss is strictly proper.